<img src="https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@bf9431e/ressources/img/logo_macmia.png" alt="Banque des Territoires · France 2030 · MACMIA" width="520">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml_v2/cours/seance1_cours.ipynb)

# Séance 4.1 — Le Machine Learning : prédire n'est pas expliquer

**Cours** · durée : 4h (2h de cours, 2h d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- définir clairement **ce que vous souhaitez prédire** et les informations utilisées pour le faire
- séparer les données entre **apprentissage** et **test** afin d'évaluer le modèle sur des observations qu'il n'a jamais vues
- entraîner un modèle puis produire des prédictions avec `fit()` et `predict()`
- mesurer la qualité des prédictions à l'aide de la **MAE**, de la **RMSE** et du **R²**
- comparer les performances du modèle à une **prédiction de référence simple**
- faire entrer une **variable qualitative** dans le modèle avec `pd.get_dummies`, et lire le coefficient d'une modalité par rapport à sa référence
- vérifier que le modèle généralise correctement, sans **surapprentissage** ni **fuite de données**

L'objectif est de comprendre le passage :

**des données → à un modèle → à des prédictions → à une évaluation de leur qualité.**

## De la régression à l'évaluation sur de nouvelles données

Dans la séance 3.4, nous avons utilisé une régression pour **décrire une relation** entre plusieurs variables.

Ici, la question change :

> **Si une nouvelle observation arrive, est-ce que notre modèle sait faire une bonne prédiction ?**

C'est cette idée de **généralisation** qui est au cœur de la séance.

> 📘 **Les bases de la régression linéaire sont dans la séance 3.4.**
>
> La séance 3.4 ne fait pas partie des séances en classe : elle est publiée
> en « pour aller plus loin ». C'est là que se trouve tout ce que cette
> séance-ci suppose acquis :
>
> - ajuster un modèle, et lire ce qu'il renvoie ;
> - lire un **coefficient**, sa **p-value** et son **intervalle de confiance** ;
> - ce que le **R²** mesure, et ce qu'il ne mesure pas ;
> - l'interprétation « **toutes choses égales par ailleurs** » ;
> - reconnaître une **extrapolation** et refuser d'y répondre.
>
> Gardez-la ouverte à côté si un point vous manque. Une différence à savoir :
> la 3.4 ajuste ses modèles avec `statsmodels` (`smf.ols`), qui affiche les
> p-values et les intervalles de confiance ; ici on utilise `scikit-learn`,
> qui est tourné vers la prédiction. Les concepts sont les mêmes, la commande
> change.
>
> [▶ Ouvrir la séance 3.4 — Régression linéaire](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc3_stats/cours/seance4_cours.ipynb)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@bf9431e/bloc4_ml/data/"

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")
cmd.head()

## 1. Quel type de problème cherche-t-on à résoudre ?

Nous disposons d'anciennes commandes dont nous connaissons :

- certaines caractéristiques ;
- le chiffre d'affaires final.

Nous voulons utiliser ces exemples pour prévoir le chiffre d'affaires d'une **nouvelle commande**.

C'est un problème d'**apprentissage supervisé** : pour les observations utilisées pour apprendre, la réponse est connue.

Deux cas principaux :

| Type de cible | Problème |
|---|---|
| un nombre (`350 €`, `12,4`, `72`) | **régression** |
| une catégorie (`oui/non`, `spam/non spam`) | **classification** |

Dans cette séance, la cible est un nombre : nous faisons donc une **régression**.

## Le cas étudié

Une ligne du fichier correspond à **une commande**.

Nous allons commencer volontairement avec un modèle très simple :

- `qte` : quantité totale commandée ;
- `ca` : chiffre d'affaires de la commande.

La question est :

> À partir de la quantité connue, quel chiffre d'affaires peut-on prévoir pour une nouvelle commande ?

In [ ]:
cmd[["qte", "ca"]].describe().round(1)

In [ ]:
cmd.plot(
    kind="scatter",
    x="qte",
    y="ca",
    alpha=0.25,
    figsize=(7, 4)
)

plt.xlabel("Quantité totale")
plt.ylabel("Chiffre d'affaires (€)")
plt.title("Quantité et chiffre d'affaires")
plt.show()

Le graphique montre une tendance positive, mais aussi beaucoup de dispersion.

Deux commandes avec la même quantité peuvent avoir des montants très différents, car les produits achetés ne sont pas les mêmes.

Notre modèle ne pourra donc pas être exact : il faudra **mesurer ses erreurs**.

## 2. `X` et `y`

Dans scikit-learn, on sépare :

- `X` : les variables utilisées pour prédire ;
- `y` : la variable que l'on cherche à prédire.

Ici :

```python
X = cmd[["qte"]]
y = cmd["ca"]
```

Scikit-learn attend X sous forme de tableau : une ligne par observation et une colonne par variable utilisée pour prédire. Même avec une seule variable (qte), on écrit donc cmd[["qte"]] avec deux paires de crochets.

In [ ]:
X = cmd[["qte"]]
y = cmd["ca"]

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

## 3. Pourquoi ne pas apprendre sur toutes les données ?

Supposons que l'on ajuste une droite sur toutes les commandes, puis que l'on mesure sa performance sur ces mêmes commandes.

Le modèle serait évalué sur des observations qu'il a déjà utilisées pour choisir ses coefficients.

Cela répond mal à la vraie question :

> **Que se passera-t-il sur une commande encore jamais vue ?**

On garde donc une partie des données de côté.

## Séparer apprentissage et test

- **train** : sert à apprendre les coefficients ;
- **test** : reste de côté et sert à évaluer le modèle à la fin.

Ici, 75 % des commandes servent à l'apprentissage et 25 % au test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=67
)

print("Apprentissage :", len(X_train), "commandes")
print("Test          :", len(X_test), "commandes")

### À quoi sert `random_state=67` ?

`train_test_split` réalise un tirage aléatoire.

Fixer `random_state` permet de retrouver **exactement le même découpage** à chaque exécution.

Le nombre `67` n'a aucune propriété statistique particulière.

## 4. Apprendre puis prédire

Le workflow est toujours le même :

```python
modele = LinearRegression()
modele.fit(X_train, y_train)
pred_test = modele.predict(X_test)
```

- `fit()` : apprend les coefficients sur les données d'apprentissage ;
- `predict()` : applique ensuite le modèle à de nouvelles valeurs de `X`.

In [ ]:
modele = LinearRegression()
modele.fit(X_train, y_train)

pred_train = modele.predict(X_train)
pred_test = modele.predict(X_test)

print("Intercept :", round(modele.intercept_, 2))
print("Coefficient de qte :", round(modele.coef_[0], 4))

Le coefficient donne la pente de la droite apprise.

S'il vaut par exemple `1.38`, cela signifie que, **dans le modèle**, une unité supplémentaire de quantité est associée à environ 1,38 € de chiffre d'affaires supplémentaire.

Cette phrase décrit une relation dans les données.  
Elle ne suffit pas à établir un **effet causal**.

## Faire une prédiction

Pour une nouvelle commande de 150 unités :

In [ ]:
nouvelle_commande = pd.DataFrame({"qte": [150]})

pred_150 = modele.predict(nouvelle_commande)[0]

print("Prédiction :", round(pred_150, 2), "€")

Le modèle fournit une **estimation**.

La bonne question n'est donc pas :

> « Est-ce que cette prédiction est exactement vraie ? »

mais plutôt :

> **« De combien le modèle se trompe-t-il en général sur des commandes qu'il n'a jamais vues ? »**

## 5. Mesurer les erreurs

Nous allons utiliser trois indicateurs complémentaires.

## MAE — Mean Absolute Error

$$\mathrm{MAE}=\frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|$$

Elle mesure la distance absolue moyenne entre la valeur réelle et la prédiction.

Une MAE de 220 € se lit :

> **« Sur le jeu de test, les prédictions s'écartent en moyenne d'environ 220 € des montants observés. »**

## RMSE — Root Mean Squared Error

$$\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}$$

Elle donne davantage de poids aux grosses erreurs.

Si la RMSE est beaucoup plus élevée que la MAE, cela suggère que quelques observations sont particulièrement mal prédites.

## R² — la part de variabilité expliquée

$$R^2=1-\frac{\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}{\sum_{i=1}^{n}(y_i-\bar{y})^2}$$

La MAE et la RMSE nous indiquent **de combien le modèle se trompe**.

Le R² répond à une question différente :

> **Les prédictions de notre modèle sont-elles meilleures qu'une prédiction très simple qui ne tient compte d'aucune variable ?**

Pour comprendre le R², commençons donc par construire un point de comparaison.

## Une prédiction de référence : la baseline

Imaginons que nous n'utilisions aucun modèle.

Une stratégie très simple serait de prédire **le même montant pour toutes les commandes** : par exemple, le chiffre d'affaires moyen observé dans les données d'apprentissage.

Si la moyenne du jeu d'apprentissage est de 530 €, on prédirait :

| Commande | Prédiction |
|---|---:|
| A | 530 € |
| B | 530 € |
| C | 530 € |
| D | 530 € |

Cette règle est volontairement simple. Elle sert uniquement de **point de comparaison**.

En machine learning, on appelle souvent ce type de référence une **baseline**.

> **Un modèle utile devrait faire mieux qu'une règle aussi simple.**

Le R² compare les erreurs de notre modèle à celles d'une prédiction constante fondée sur la moyenne.

## Comment interpréter le R² ?

Repères :

- R² = 1 : prédictions parfaites ;
- R² = 0 : pas mieux, en termes de somme des carrés, que la moyenne ;
- R² < 0 : pire que cette référence.


Par exemple, si R² = 0.70, alors les erreurs au carré du modèle représentent environ **30 %** de celles de la référence.

Autrement dit, le modèle réduit d'environ **70 % les erreurs quadratiques** par rapport à cette référence.

**Attention :** R² = 0.70 ne signifie pas « 70 % des commandes sont bien prédites ».

In [ ]:
# La baseline : predire toujours la meme valeur, la moyenne d'apprentissage
pred_baseline = np.full(len(y_test), y_train.mean())
mae_baseline = mean_absolute_error(y_test, pred_baseline)

print("Prediction de reference :", round(y_train.mean(), 1), "euros pour toutes")

In [ ]:
mae = mean_absolute_error(y_test, pred_test)
rmse = np.sqrt(mean_squared_error(y_test, pred_test))
r2 = r2_score(y_test, pred_test)

resultats = pd.Series({
    "MAE (€)": mae,
    "RMSE (€)": rmse,
    "R²": r2
})

resultats.round(3)

In [ ]:
print("MAE baseline :", round(mae_baseline, 1), "€")
print("MAE modèle   :", round(mae, 1), "€")

## Les trois métriques ne racontent pas la même chose

- **MAE** : taille moyenne des erreurs dans une unité concrète ;
- **RMSE** : attire l'attention sur les erreurs importantes ;
- **R²** : positionne le modèle par rapport à une référence.

Dans un problème métier, il est souvent utile de regarder au moins une mesure **dans l'unité de la cible**, comme la MAE.

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_test, pred_test, alpha=0.35)

borne = max(y_test.max(), pred_test.max())
plt.plot([0, borne], [0, borne], linestyle="--")

plt.xlabel("Montant observé (€)")
plt.ylabel("Montant prédit (€)")
plt.title("Observé et prédit sur le jeu de test")
plt.show()

Sur la diagonale, la prédiction serait exacte.

Plus un point s'éloigne de la diagonale, plus l'erreur est importante.

Un score moyen ne remplace donc pas l'examen de la distribution des erreurs.

## 6. Le modèle généralise-t-il ?

On peut comparer ses performances sur :

- les commandes utilisées pour apprendre ;
- les commandes du test.

In [ ]:
scores = pd.DataFrame(
    {
        "apprentissage": [
            mean_absolute_error(y_train, pred_train),
            np.sqrt(mean_squared_error(y_train, pred_train)),
            r2_score(y_train, pred_train),
        ],
        "test": [
            mean_absolute_error(y_test, pred_test),
            np.sqrt(mean_squared_error(y_test, pred_test)),
            r2_score(y_test, pred_test),
        ],
    },
    index=["MAE", "RMSE", "R²"],
)

scores.round(3)

### Comment lire l'écart train / test ?

| Situation | Lecture possible |
|---|---|
| train et test bons et proches | généralisation encourageante |
| train très bon, test nettement moins bon | surapprentissage possible |
| train et test tous deux faibles | variables ou modèle peu informatifs |

Le surapprentissage signifie que le modèle s'adapte très bien aux données connues, mais beaucoup moins bien à de nouvelles observations.

## 7. Ajouter une variable qualitative

Notre modèle ne connaît qu'une chose : la quantité commandée. Il traite de la même façon une commande de 150 unités passée depuis l'Irlande et la même commande passée depuis le Royaume-Uni, alors que les paniers n'y sont pas les mêmes.

Le fichier contient cette information, dans la colonne `pays`. Mais un modèle **multiplie des nombres par des coefficients** : il ne sait rien faire de la chaîne `"Irlande"`.

La solution standard est l'**encodage indicatrice** : une colonne par modalité, qui vaut 1 quand la commande vient de ce pays et 0 sinon. `pd.get_dummies` les fabrique toutes d'un coup.

In [ ]:
pays = pd.get_dummies(cmd["pays"], prefix="pays", drop_first=True)

print(pays.shape[1], "colonnes pour", cmd["pays"].nunique(), "pays")
pays[["pays_France", "pays_Irlande", "pays_Royaume-Uni"]].head(3)

Chaque ligne porte au plus un `True` : celui de son pays. Pour le modèle, `True` vaut 1 et `False` vaut 0.

### Pourquoi `drop_first=True` ?

23 pays, mais 22 colonnes : la première modalité dans l'ordre alphabétique, `Allemagne`, n'a pas la sienne. Ce n'est pas une perte. Une commande dont les 22 colonnes valent 0 vient forcément d'Allemagne — l'information y est déjà. La colonne supplémentaire serait redondante et rendrait les coefficients instables.

Cette modalité devient la **référence** : les autres coefficients se lisent *par rapport à elle*.

### Assembler, puis réapprendre

Les colonnes quantitatives et les indicatrices se recollent avec `pd.concat(..., axis=1)`. Le reste du workflow ne change pas : on redécoupe, on ajuste, on prédit.

> ⚠️ **Gardez le même `random_state=67`**, sinon les deux modèles ne sont plus évalués sur les mêmes commandes et la comparaison ne veut rien dire.

In [ ]:
X2 = pd.concat([cmd[["qte"]], pays], axis=1)

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.25, random_state=67
)

modele2 = LinearRegression().fit(X2_train, y2_train)
pred2_test = modele2.predict(X2_test)

In [ ]:
comparaison = pd.DataFrame(
    {
        "qte seule": [mae, rmse, r2],
        "qte + pays": [
            mean_absolute_error(y2_test, pred2_test),
            np.sqrt(mean_squared_error(y2_test, pred2_test)),
            r2_score(y2_test, pred2_test),
        ],
    },
    index=["MAE (€)", "RMSE (€)", "R²"],
)

comparaison.round(3)

La MAE passe de 221,2 € à 217,8 € et le R² de 0,702 à 0,719. Le gain est réel, et il est modeste : le pays explique une petite partie de ce que la quantité laissait de côté.

### Lire le coefficient d'une modalité

Un coefficient d'indicatrice est un **écart au pays de référence, à quantité égale**.

In [ ]:
coefs = pd.Series(modele2.coef_, index=X2.columns).round(1)

coefs[["qte", "pays_France", "pays_Irlande", "pays_Royaume-Uni"]]

`pays_Irlande` vaut +98,3 : à quantité identique, une commande irlandaise est prédite environ 98 € au-dessus d'une commande allemande. `pays_Royaume-Uni` vaut −110,1, soit 110 € en dessous.

Deux précautions avant de faire entrer n'importe quelle colonne de texte :

- **Le nombre de colonnes suit le nombre de modalités.** Une variable à 300 modalités en ajoute 299, souvent plus que le modèle n'a d'observations pour les estimer sérieusement.
- **Une modalité rare donne un coefficient peu fiable.** Le Canada et les États-Unis comptent une commande chacun : leur coefficient est ajusté sur une seule ligne. D'où la modalité `Autre` de ce fichier, qui regroupe les pays trop peu représentés.

## 8. Fuite de données : un bon score peut être trompeur

Une variable ne doit être utilisée que si elle sera **disponible au moment réel de la prédiction**.

Posez toujours la question :

> **« Est-ce que je connaîtrai cette information au moment où je devrai prédire ? »**

Exemple extrême : si `ca` se trouve dans `X`, on donne directement la réponse au modèle.

In [ ]:
X_fuite = cmd[["qte", "ca"]]
y_fuite = cmd["ca"]

X_tr_f, X_te_f, y_tr_f, y_te_f = train_test_split(
    X_fuite,
    y_fuite,
    test_size=0.25,
    random_state=67
)

modele_fuite = LinearRegression()
modele_fuite.fit(X_tr_f, y_tr_f)

pred_fuite = modele_fuite.predict(X_te_f)

print("R² test :", round(r2_score(y_te_f, pred_fuite), 6))

Un score parfait ou presque parfait doit donc déclencher une vérification avant toute célébration :

1. la cible est-elle directement dans `X` ?
2. une variable a-t-elle été calculée après l'événement que l'on veut prévoir ?
3. des informations du test ont-elles servi pendant l'apprentissage ?

## 9. Le découpage doit ressembler à l'usage réel

Le découpage aléatoire est utile quand les observations sont raisonnablement interchangeables.

Mais si l'objectif est de prévoir **le futur**, une évaluation chronologique peut être plus réaliste :

- apprendre sur les mois passés ;
- tester sur une période ultérieure.

Un bon protocole d'évaluation cherche à reproduire la situation dans laquelle le modèle sera réellement utilisé.

## Synthèse

Le workflow de base d'un problème de régression prédictive est :

```text
définir X et y
      ↓
séparer train / test
      ↓
fit sur le train
      ↓
predict sur le test
      ↓
comparer aux valeurs réelles
      ↓
interpréter les erreurs
```

Le point essentiel n'est pas d'obtenir le score le plus élevé possible.

Il faut surtout vérifier que la performance est mesurée **sur de nouvelles observations**, avec des variables réellement disponibles au moment de la prédiction.

### Aide-mémoire des commandes

| Vous voulez... | La commande |
|---|---|
| découper les données | `train_test_split(X, y, test_size=0.25, random_state=67)` |
| transformer une colonne de texte en 0/1 | `pd.get_dummies(cmd["pays"], prefix="pays", drop_first=True)` |
| recoller quantitatif et indicatrices | `pd.concat([cmd[["qte"]], pays], axis=1)` |
| ajuster un modèle | `m = LinearRegression().fit(X_train, y_train)` |
| prédire | `m.predict(X_test)` |
| l'erreur moyenne, en euros | `mean_absolute_error(y_test, p)` |
| l'erreur qui punit les grosses fautes | `mean_squared_error(y_test, p) ** 0.5` |
| la part expliquée | `r2_score(y_test, p)` |

### Les quatre phrases à retenir

1. **Un modèle noté sur les données qui l'ont produit se note lui-même.**
   La seule note qui compte est celle obtenue sur des lignes qu'il n'a jamais vues.

2. **RMSE et MAE se lisent en euros.** « Je me trompe de 221 € en moyenne » est
   une phrase de gestion ; « R² = 0,70 » n'en est pas une.

3. **Un écart entre apprentissage et test est normal ; un écart béant est un
   surapprentissage.** Ici l'écart de R² vaut 0,023 : la régression généralise.
   Quand il se creuse, le modèle a retenu au lieu d'apprendre.

4. **Une variable qui contient la réponse donne un modèle parfait et inutile.**